### Getting data

In [1]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

--2026-10-27 16:15:29--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Loaded CA certificate '/etc/ssl/certs/ca-certificates.crt'
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8003::154, 2606:50c0:8000::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv’

car_fuel_efficiency 100%[===================>] 620.29K  1.30MB/s    in 0.5s    

2026-10-27 16:15:30 (1.30 MB/s) - ‘car_fuel_efficiency_2026.csv’ saved [635180/635180]



### Preparing the dataset

In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv("car_fuel_efficiency_2026.csv")

In [3]:
data.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
data.columns

Index(['model_year', 'origin', 'fuel_type', 'drivetrain', 'num_doors',
       'engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration', 'fuel_efficiency_mpg'],
      dtype='str')

In [5]:
data.dtypes

model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

In [6]:
data.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [7]:
data = data.fillna(0.0)

In [8]:
data.isnull().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
df_full_train, df_test = train_test_split(data, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

df_train = df_train.reset_index()
df_val = df_val.reset_index()
df_test = df_test.reset_index()

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [11]:
from sklearn.feature_extraction import DictVectorizer

In [12]:
train_dicts = df_train.to_dict(orient='records')

dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(train_dicts)

### Question 1

Let's train a decision tree regressor to predict the `fuel_efficiency_mpg` variable. 

- Train a model with `max_depth=1`.

Which original feature is used for splitting the data? If you use `DictVectorizer`, map a one-hot feature such as `origin=USA` back to its
original column name.

- `'vehicle_weight'`
- `'model_year'`
- `'origin'`
- `'fuel_type'`

In [13]:
from sklearn.tree import DecisionTreeRegressor, export_text

In [14]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with 

In [15]:
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



### Question 2

Train a random forest regressor with these parameters:

- `n_estimators=10`
- `random_state=1`
- `n_jobs=-1` (optional - to make training faster)

What's the RMSE of this model on the validation data?

- 0.1837
- 1.837
- 18.37
- 183.7

In [16]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [17]:
val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [18]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)

root_mean_squared_error(y_val, y_pred)

1.8218156877137708

### Question 3

Now let's experiment with the `n_estimators` parameter

- Try these values of this parameter: `[10, 50, 100, 150]`.
- Set `random_state` to `1`.
- Evaluate the model on the validation dataset.

Which value gives the lowest validation RMSE? Consider 3 decimal places for
calculating the answer.

- 10
- 50
- 100
- 150


In [19]:
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)

    rmse = root_mean_squared_error(y_val, y_pred)
    print(f"n: {n}, rmse: {round(rmse, 3)}")

n: 10, rmse: 1.822
n: 50, rmse: 1.768
n: 100, rmse: 1.761
n: 150, rmse: 1.755


### Question 4

Let's select the best `max_depth`:

- Try different values of `max_depth`: `[10, 15, 20, 25]`
- For each of these values,
  - try different values of `n_estimators` from `[10, 50, 100, 150]`
  - calculate the mean RMSE
- Fix the random seed: `random_state=1`

What's the best `max_depth`, using the mean RMSE?

- 10
- 15
- 20
- 25


In [20]:
for d in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1)
        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)

        rmse = root_mean_squared_error(y_val, y_pred)
        scores.append(rmse)
        
    mean_rmse = np.mean(scores)

    print(f"d: {d}, n: {n}, rmse: {round(mean_rmse, 3)}")

d: 10, n: 150, rmse: 1.76
d: 15, n: 150, rmse: 1.781
d: 20, n: 150, rmse: 1.781
d: 25, n: 150, rmse: 1.777


### Question 5

We can extract feature importance information from tree-based models.

At each step of the decision tree learning algorithm, it finds the best split.
When doing it, we can calculate "gain" - the reduction in impurity before and after the split.
This gain is quite useful in understanding what are the important features for tree-based models.

In Scikit-Learn, tree-based models contain this information in the
[`feature_importances_`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html#sklearn.ensemble.RandomForestRegressor.feature_importances_)
field.

For this homework question, we'll find the most important feature:

- Train the model with these parameters:
  - `n_estimators=10`,
  - `max_depth=20`,
  - `random_state=1`,
  - `n_jobs=-1` (optional)
- Get the feature importance information from this model

What's the most important feature (among these 4)?

- `vehicle_weight`
- `horsepower`
- `acceleration`
- `engine_displacement`

In [21]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

feature_importance = pd.DataFrame({
    'feature': dv.get_feature_names_out(),
    'importance': rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='importance',
    ascending=False
)

print(feature_importance)

                         feature  importance
10                    model_year    0.288734
16                vehicle_weight    0.193563
6             fuel_type=Gasoline    0.160669
8                     horsepower    0.068320
9                          index    0.055143
4            engine_displacement    0.052575
0                   acceleration    0.043980
15                    origin=USA    0.041739
7               fuel_type=Hybrid    0.024422
2   drivetrain=Front-wheel drive    0.019536
12                     num_doors    0.015855
5               fuel_type=Diesel    0.008289
1     drivetrain=All-wheel drive    0.007280
13                   origin=Asia    0.006404
11                 num_cylinders    0.005926
3    drivetrain=Rear-wheel drive    0.004467
14                 origin=Europe    0.003099


### Question 6

Now let's train an XGBoost model! For this question, we'll tune the `eta` parameter:

- Install XGBoost
- Create DMatrix for train and validation
- Create a watchlist
- Train a model with these parameters for 100 rounds:

```
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}
```

Now change `eta` from `0.3` to `0.1`.

Which eta leads to the best RMSE score on the validation dataset?

- 0.3
- 0.1
- Both give equal value

In [22]:
import xgboost as xgb

In [23]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

watchlist = [(dval, 'eval'), (dtrain, 'train')]

In [24]:
for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
    
        'objective': 'reg:squarederror',
        'nthread': 8,
    
        'seed': 1,
        'verbosity': 1,
    }

    model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist)
    
    y_pred = model.predict(dval)
    rmse = root_mean_squared_error(y_val, y_pred)
    
    print(f'eta={eta}: RMSE={rmse:.4f}')

[0]	eval-rmse:2.46782	train-rmse:2.40653
[1]	eval-rmse:2.15839	train-rmse:2.07094
[2]	eval-rmse:1.97618	train-rmse:1.86435
[3]	eval-rmse:1.86963	train-rmse:1.73421
[4]	eval-rmse:1.80597	train-rmse:1.64795
[5]	eval-rmse:1.77376	train-rmse:1.59090
[6]	eval-rmse:1.74953	train-rmse:1.55250
[7]	eval-rmse:1.73888	train-rmse:1.52398
[8]	eval-rmse:1.73314	train-rmse:1.50219
[9]	eval-rmse:1.73026	train-rmse:1.48207
[10]	eval-rmse:1.72898	train-rmse:1.46697
[11]	eval-rmse:1.73136	train-rmse:1.45332
[12]	eval-rmse:1.73284	train-rmse:1.44176
[13]	eval-rmse:1.73243	train-rmse:1.43169
[14]	eval-rmse:1.73421	train-rmse:1.42431
[15]	eval-rmse:1.73606	train-rmse:1.41322
[16]	eval-rmse:1.73651	train-rmse:1.40950
[17]	eval-rmse:1.73752	train-rmse:1.40398
[18]	eval-rmse:1.74031	train-rmse:1.38761
[19]	eval-rmse:1.74466	train-rmse:1.37482
[20]	eval-rmse:1.74516	train-rmse:1.36903
[21]	eval-rmse:1.74575	train-rmse:1.36528
[22]	eval-rmse:1.74607	train-rmse:1.36216
[23]	eval-rmse:1.74731	train-rmse:1.35697
[2

eta=0.3: RMSE=1.8404\
eta=0.1: RMSE=1.7304